# Getting started with the HHO_1D solver

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

The module `HHO_kernel` in this repository provides an implementation of the HHO solver of the Poisson equation
$$
-u'' = f
$$
on an interval of $\mathbb{R}$.
The class implementing this solver is called `HHO_poisson`, so let us start and import it right away.

In [ ]:
from HHO_kernel import HHO_poisson

As a first example, we solve the equation with the exact solution $u(x) = \cos(8x) + x^2$, so $f(x) = 64 \cos(8x) - 2$, on the interval $(0,\pi)$. At the boundary, we show how to impose a homogeneous Neumann condition at $x=0$ and the Dirichlet condition $u(\pi) = 1 + \pi^2$ at the right.

We define a homogeneous grid consisting of 9 cells to discretize the equation.

In [ ]:
x_left = 0.0
x_right = np.pi
xx = np.linspace(x_left, x_right, 10)

To initialize the solver, we need the grid and the polynomial degree of the cell unknowns of the HHO approximation. The default value is 0, but we can specify another value through the `degree` argument at initialization. We shall use degree 2 on the cells here.

In [ ]:
poisson = HHO_poisson(xx, degree=2)

The next step is to specify the boundary conditions for our problem. This is done by first setting the *type* of boundary conditions to be applied. These are stored in the `boundary_conditions` attribute of the `poisson` solver, that can be directly set by the user. It must be of the form `'XX'` with each `X` being either `D` (for Dirichlet) or `N` (for Neumann). The first `X` sets the boundary condition on the left of the interval, and the second `X` on the right.

Dirichlet boundary conditions can be inhomogeneous, but the solver only takes homogeneous Neumann conditions.

For our problem, the boundary conditions are `'ND'`.

In [ ]:
poisson.boundary_conditions = 'ND'

## Solving the Poisson equation

A boundary-value can now be solved by calling the `poisson.solve()` method. It takes the following arguments:
* The function $f$ on the right-hand side of the Poisson equation (operating on `numpy` arrays).
* The values of the boundary conditions as optional keyword arguments:
    * `bc_left` provides the value of the Dirichlet condition on the left when `boundary conditions` is of the form `'DX'`.
    * `bc_right` provides the value of the Dirichlet condition on the right when `boundary conditions` is of the form `'XD'`.
    * When `boundary_conditions` is set to `NN`, the average must be provided to obtain a unique solution, which is done through the keyword argument `average`.

Our equation is thus solved as follows.

In [ ]:
f = lambda x: 64*np.cos(8*x) - 2
bc_right = 1+np.pi**2
poisson.solve(f, bc_right=bc_right)

`solve()` computes the unknowns of the HHO method in the discrete HHO space.
These consist of 1 value for every face (or grid point in 1D), and the $L^2$-projection of the cell unknowns on a polynomial basis, which in our case consists of 3 basis functions per cell, since the polynomial degree of the unknowns is 2. More information on the choice of basis below.

These unknowns corresponding to our approximation are now stored in the `poisson` solver. 
The values at the faces are stored in the `solution_face` attribute of the solver.

In [ ]:
print(f'HHO unknowns at the faces: {poisson.solution_face}')

The cells are represented by the `cells` attribute, and the solution on cell `i` is stored in `poisson.cells[i].solution`.

In [ ]:
for i, cell in enumerate(poisson.cells):
    print(f'HHO unknowns on cell {i}: {np.round(cell.solution,2)}')

The reconstruction of the potential $u$ in each cell is a polynomial of degree 3. It is also stored in each cell through its $L^2$-projection on the basis, which consists of 4 functions for the reconstruction in this example.

In [ ]:
for i, cell in enumerate(poisson.cells):
    print(f'L2 projection of the reconstruction on cell {i}: {np.round(cell.solution_reconstruction,2)}')

## Visualizing the solution

The solver comes equipped with a method `plot()` to visualize the HHO approximation. To use it we pass:
* A `matplotlib.axes.Axes` object for the plot to be displayed on.
* A number of optional arguments to specify what we want to visualize:
    * "faces" for the DOFs of the HHO approximation at the faces.
    * "cells" for the DOFs of the HHO approximation on the cells.
    * "reconstruction" for the reconstruction of the potential $u$ based on the DOFs that result from the discrete problem that was solved by `poisson.solve()`.

Specifying nothing plots all the above options.

In [ ]:
# Plot of the DOFs of the HHO approximation
xx_plot = np.linspace(x_left, x_right, 100)
fig, axs = plt.subplots(1,2,figsize=(10,4))
poisson.plot(axs[0], "faces", "cells")
axs[0].legend()
# Plot the reconstruction for the potential and compare it with u
u = lambda x: np.cos(8*x) + x**2
axs[1].plot(xx_plot, u(xx_plot), 'black', label="$u(x)$")
poisson.plot(axs[1], "faces", "reconstruction")
axs[1].legend();

We clearly see from the plots that no continuity is imposed on neighboring cell and face unknowns. The reconstruction, on the other hand, equals the face unknowns at all the faces and is thus continuous. This is a particular property of the HHO method in 1D when $k\geq1$.

Let us quickly illustrate how we can use the `poisson` solver to solve the case $f(x) = 16*\cos(4x)$, $u'(0)=0$, $u'(\pi)=0$, $\int_0^\pi u = 0$, which corresponds to the exact solution $u(x) = \cos(x)$.

In [ ]:
poisson.boundary_conditions = 'NN'
poisson.solve(lambda x: 16*np.cos(4*x), average=0)
fig, ax = plt.subplots(1,1)
poisson.plot(ax)
ax.legend();

## The polynomial basis on the cell

We now focus on changing the polynomial degree. It is important to say a few words about the basis implementation here. We have implemented a monomial basis, which is not and the basis of $L^2$-orthogonal Legendre polynomials, which is accessed through the `scipy.special` module. The default one used is the monomial basis.

Because the monomial basis is not orthogonal, it can be dangerous to work with it numerically for high polynomial degrees.
It is therefore important to activate the `orthonormal_basis=True` option in the initializer of the `HHO_poisson` solver in the next example.
When this is omitted, the example of a 19th order HHO approximation leads to a singular matrix for the global problem on my computer.

In [ ]:
degree = 18
xx = np.linspace(x_left,x_right,3) # same domain as before, but a very coarse grid
poisson = HHO_poisson(xx, degree) # build a new solver
poisson.boundary_conditions = 'ND'
poisson.solve(f, bc_right=bc_right) # we return to the first example

Changing to the Legendre basis can be done through the `basis` argument at initialization of the solver. There is no need to activate the orthonormalization procedure of the solver in this case. It runs a bit slower, however, as can be observed in the validation notebooks in this repository which carries out some larger computations.

In [ ]:
poisson = HHO_poisson(xx, degree, basis="legendre") # build a new solver
poisson.boundary_conditions = 'ND'
poisson.solve(f, bc_right=bc_right)

This time there is no `LinAlgError`, and we have succeeded in computing a discrete approximation. We can again now compare the reconstruction of the HHO solution to the function $u$.

In [ ]:
# Plot of the DOFs of the HHO approximation
xx_plot = np.linspace(x_left, x_right, 100)
fig, axs = plt.subplots(1,2,figsize=(10,4))
poisson.plot(axs[0], "faces", "cells")
axs[0].legend()
# Plot the reconstruction for the potential and compare it with u
u = lambda x: np.cos(8*x) + x**2
axs[1].plot(xx_plot, u(xx_plot), 'black', label="$u(x)$")
poisson.plot(axs[1], "faces", "reconstruction")
axs[1].legend();

Even with only two cells, the high-order approximation is a good approximation of the solution $u$. We even notice that the difference between the cell unknowns and the potential reconstruction has become rather small, and that the jump between the cell unknowns is undetectable on the scale of the plot.